<a href="https://colab.research.google.com/github/hhw215/Computer-Vision-Project/blob/main/cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
import sys
!pip install -U timm pandas pillow numpy torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/42

In [ ]:
import copy
import csv
import json
import random
import time
from collections.abc import Iterable
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

In [ ]:
from google.drive import mount
mount('/content/drive')

# Globals

In [ ]:
BASE_DIR = "/content/drive/MyDrive/cv"
CONFIG = {
    "seed": 42,
    "image_size": 224,
    "model_name": "deit_small_patch16_224",
    "pretrained": True,
    "attention_type": "standard",
    "replace_layers": "none",
    "top_k": 4,
    "batch_size": 32,
    "num_workers": 2,
    "epochs": 10,
    "lr": 1e-4,
    "router_lr": 3e-4,
    "weight_decay": 0.05,
    "warmup_ratio": 0.05,
    "min_lr": 1e-6,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "train_csv": f"{BASE_DIR}/dataset/splits/train.csv",
    "val_csv": f"{BASE_DIR}/dataset/splits/val.csv",
    "test_csv": f"{BASE_DIR}/dataset/splits/test.csv",
    "images_root": f"{BASE_DIR}/dataset/raw",
    "run_name": "template_run",
    "out_dir": f"{BASE_DIR}/outputs",
}
print(json.dumps(CONFIG, indent=2))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

{
  "seed": 42,
  "image_size": 224,
  "model_name": "deit_small_patch16_224",
  "pretrained": true,
  "attention_type": "standard",
  "replace_layers": "none",
  "top_k": 4,
  "batch_size": 32,
  "num_workers": 2,
  "epochs": 10,
  "lr": 0.0001,
  "router_lr": 0.0003,
  "weight_decay": 0.05,
  "warmup_ratio": 0.05,
  "min_lr": 1e-06,
  "device": "cpu",
  "train_csv": "/content/drive/MyDrive/cv/dataset/splits/train.csv",
  "val_csv": "/content/drive/MyDrive/cv/dataset/splits/val.csv",
  "test_csv": "/content/drive/MyDrive/cv/dataset/splits/test.csv",
  "images_root": "/content/drive/MyDrive/cv/dataset/raw",
  "run_name": "template_run",
  "out_dir": "/content/drive/MyDrive/cv/outputs"
}
CUDA available: False


# Utils

In [ ]:
REQUIRED_TRIPLET_COLUMNS = (
    "reference",
    "candidate_a",
    "candidate_b",
    "choice",
)
class NightsTripletDataset(Dataset):
    def __init__(self, split_csv: str, images_root: Optional[str] = None, transform=None) -> None:
        self.split_csv = Path(split_csv)
        self.images_root = Path(images_root) if images_root else None
        self.transform = transform
        df = pd.read_csv(self.split_csv)
        missing = [column for column in REQUIRED_TRIPLET_COLUMNS if column not in df.columns]
        if missing:
            raise ValueError(f"Split CSV is missing required columns {missing}. Found: {list(df.columns)}")
        self.df = df
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        ref_path = self._resolve_path(row["reference"])
        a_path = self._resolve_path(row["candidate_a"])
        b_path = self._resolve_path(row["candidate_b"])
        ref_img = self._load_image(ref_path)
        a_img = self._load_image(a_path)
        b_img = self._load_image(b_path)
        if self.transform is not None:
            ref_img = self.transform(ref_img)
            a_img = self.transform(a_img)
            b_img = self.transform(b_img)
        label = int(row["choice"])
        return {
            "reference": ref_img,
            "candidate_a": a_img,
            "candidate_b": b_img,
            "choice": label,
            "reference_path": str(ref_path),
            "candidate_a_path": str(a_path),
            "candidate_b_path": str(b_path),
        }
    def _resolve_path(self, raw_path: str) -> Path:
        path = Path(str(raw_path))
        if path.is_absolute():
            return path
        if self.images_root is None:
            return path
        return self.images_root / path
    @staticmethod
    def _load_image(path: Path) -> Image.Image:
        with Image.open(path) as image:
            return image.convert("RGB")
    def check_missing_files(self) -> pd.DataFrame:
        rows = []
        for index, row in self.df.iterrows():
            ref_path = self._resolve_path(row["reference"])
            a_path = self._resolve_path(row["candidate_a"])
            b_path = self._resolve_path(row["candidate_b"])
            if not (ref_path.exists() and a_path.exists() and b_path.exists()):
                rows.append({
                    "row_index": index,
                    "reference_exists": ref_path.exists(),
                    "candidate_a_exists": a_path.exists(),
                    "candidate_b_exists": b_path.exists(),
                })
        return pd.DataFrame(rows)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
def build_transforms(image_size: int = 224, train: bool = True) -> transforms.Compose:
    resize_size = int((256 / 224) * image_size)
    if train:
        return transforms.Compose([
            transforms.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
            transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
DEFAULT_BACKBONES: Sequence[str] = (
    "deit_tiny_patch16_224",
    "deit_small_patch16_224",
    "deit_base_patch16_224",
)
@dataclass(frozen=True)
class BackboneConfig:
    model_name: str = "deit_small_patch16_224"
    pretrained: bool = True
    image_size: int = 224
    proj_dim: Optional[int] = None
    drop_path_rate: float = 0.0
    l2_normalize: bool = True
class VisionTransformerEncoder(nn.Module):
    def __init__(self, config: BackboneConfig) -> None:
        super().__init__()
        self.config = config
        self.backbone = timm.create_model(config.model_name, pretrained=config.pretrained, num_classes=0, img_size=config.image_size, drop_path_rate=config.drop_path_rate)
        self.embedding_dim = int(self.backbone.num_features)
        output_dim = config.proj_dim or self.embedding_dim
        self.projection = nn.Identity() if output_dim == self.embedding_dim else nn.Linear(self.embedding_dim, output_dim)
        self.output_dim = output_dim
    def forward_features(self, images: torch.Tensor) -> torch.Tensor:
        features = self.backbone.forward_features(images)
        if isinstance(features, (tuple, list)):
            features = features[0]
        if features.ndim == 3:
            embeddings = features[:, 0]
        elif features.ndim == 2:
            embeddings = features
        else:
            raise ValueError(f"Unsupported feature shape: {tuple(features.shape)}")
        embeddings = self.projection(embeddings)
        if self.config.l2_normalize:
            embeddings = F.normalize(embeddings, dim=-1)
        return embeddings
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.forward_features(images)
def list_vit_backbones() -> Sequence[str]:
    return DEFAULT_BACKBONES
AttentionFactory = Callable[[nn.Module, int], nn.Module]
def _resolve_vit_blocks(model: nn.Module):
    backbone = getattr(model, "backbone", model)
    blocks = getattr(backbone, "blocks", None)
    if blocks is None:
        raise AttributeError("Model does not expose a ViT-style 'blocks' attribute.")
    return blocks
def replace_attention_modules(model: nn.Module, attention_factory: AttentionFactory, block_indices: Optional[Iterable[int]] = None) -> List[str]:
    blocks = _resolve_vit_blocks(model)
    if block_indices is None:
        block_indices = range(len(blocks))
    replaced_paths: List[str] = []
    for block_index in block_indices:
        old_attention = blocks[block_index].attn
        blocks[block_index].attn = attention_factory(old_attention, block_index)
        replaced_paths.append(f"blocks.{block_index}.attn")
    return replaced_paths
class MoHAttention(nn.Module):
    def __init__(self, old_attn: nn.Module, top_k: int | None = 4):
        super().__init__()
        self.num_heads = old_attn.num_heads
        self.head_dim = old_attn.head_dim
        self.attn_dim = old_attn.attn_dim
        self.scale = old_attn.scale
        self.qkv = copy.deepcopy(old_attn.qkv)
        self.q_norm = copy.deepcopy(old_attn.q_norm)
        self.k_norm = copy.deepcopy(old_attn.k_norm)
        self.attn_drop = copy.deepcopy(old_attn.attn_drop)
        self.norm = copy.deepcopy(old_attn.norm)
        self.proj = copy.deepcopy(old_attn.proj)
        self.proj_drop = copy.deepcopy(old_attn.proj_drop)
        dim = old_attn.qkv.in_features
        self.router = nn.Linear(dim, self.num_heads)
        self.top_k = top_k
    def forward(self, x, attn_mask=None, is_causal=False):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv(x).reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        head_out = attn @ v
        route_logits = self.router(x.mean(dim=1))
        if self.top_k is not None and self.top_k < self.num_heads:
            topk_vals, topk_idx = route_logits.topk(self.top_k, dim=-1)
            masked_logits = route_logits.new_full(route_logits.shape, float("-inf"))
            masked_logits.scatter_(1, topk_idx, topk_vals)
            route_logits = masked_logits
        head_weights = route_logits.softmax(dim=-1)
        head_out = head_out * head_weights[:, :, None, None]
        out = head_out.transpose(1, 2).reshape(batch_size, seq_len, self.attn_dim)
        out = self.norm(out)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out
def parse_replace_layers(value: str) -> str | List[int] | None:
    lowered = value.strip().lower()
    if lowered in {"none", "", "null"}:
        return None
    if lowered == "all":
        return "all"
    return [int(item.strip()) for item in value.split(",") if item.strip()]
def resolve_block_indices(num_blocks: int, replace_layers: str | Iterable[int] | None) -> List[int]:
    if replace_layers is None:
        return []
    if replace_layers == "all":
        return list(range(num_blocks))
    return list(replace_layers)
def make_attention_factory(attention_type: str, top_k: int | None = 4):
    if attention_type == "standard":
        return lambda old_attn, block_idx: old_attn
    if attention_type == "moh":
        return lambda old_attn, block_idx: MoHAttention(old_attn, top_k=top_k)
    if attention_type == "pyra":
        return lambda old_attn, block_idx: PyraAttention(old_attn, top_k=top_k)
    if attention_type == "meta":
        return lambda old_attn, block_idx: MetaAttention(old_attn, top_k=top_k)
    raise ValueError(f"Unknown attention_type: {attention_type}")
@dataclass
class TripletScores:
    sim_a: torch.Tensor
    sim_b: torch.Tensor
    pred: torch.Tensor
class DreamSimLikePipeline(nn.Module):
    def __init__(self, encoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder
    def embed(self, images: torch.Tensor) -> torch.Tensor:
        return self.encoder(images)
    def score_triplet(self, reference: torch.Tensor, candidate_a: torch.Tensor, candidate_b: torch.Tensor) -> TripletScores:
        e_ref = self.embed(reference)
        e_a = self.embed(candidate_a)
        e_b = self.embed(candidate_b)
        sim_a = F.cosine_similarity(e_ref, e_a, dim=-1)
        sim_b = F.cosine_similarity(e_ref, e_b, dim=-1)
        pred = torch.where(sim_a >= sim_b, 0, 1).long()
        return TripletScores(sim_a=sim_a, sim_b=sim_b, pred=pred)
    def forward(self, reference: torch.Tensor, candidate_a: torch.Tensor, candidate_b: torch.Tensor) -> Dict[str, torch.Tensor]:
        out = self.score_triplet(reference, candidate_a, candidate_b)
        return {"sim_a": out.sim_a, "sim_b": out.sim_b, "pred": out.pred}
def evaluate(split_csv: str, images_root: str, model_name: str, pretrained: bool, image_size: int, batch_size: int, num_workers: int, device: str, max_batches: int | None = None, encoder: torch.nn.Module | None = None) -> Dict[str, float]:
    transform = build_transforms(image_size=image_size, train=False)
    dataset = NightsTripletDataset(split_csv=split_csv, images_root=images_root, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=device.startswith("cuda"))
    if encoder is None:
        encoder = VisionTransformerEncoder(BackboneConfig(model_name=model_name, pretrained=pretrained, image_size=image_size))
    encoder = encoder.to(device)
    encoder.eval()
    pipeline = DreamSimLikePipeline(encoder).to(device)
    pipeline.eval()
    total = 0
    correct = 0
    elapsed_seconds = 0.0
    batches_ran = 0
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            ref = batch["reference"].to(device, non_blocking=True)
            cand_a = batch["candidate_a"].to(device, non_blocking=True)
            cand_b = batch["candidate_b"].to(device, non_blocking=True)
            label = batch["choice"].to(device, non_blocking=True)
            start = time.perf_counter()
            out = pipeline(ref, cand_a, cand_b)
            if device.startswith("cuda"):
                torch.cuda.synchronize()
            elapsed_seconds += time.perf_counter() - start
            pred = out["pred"]
            correct += (pred == label).sum().item()
            total += label.numel()
            batches_ran += 1
    if total == 0:
        raise RuntimeError("No evaluation samples available. Check split CSV and image paths.")
    accuracy = correct / total
    images_processed = total * 3
    ms_per_image = (elapsed_seconds * 1000.0 / images_processed) if images_processed > 0 else float("nan")
    images_per_second = (images_processed / elapsed_seconds) if elapsed_seconds > 0 else float("inf")
    metrics = {
        "samples": total,
        "batches": batches_ran,
        "accuracy_2afc": accuracy,
        "ms_per_image": ms_per_image,
        "images_per_second": images_per_second,
        "elapsed_seconds": elapsed_seconds,
    }
    if device.startswith("cuda"):
        metrics["max_vram_mb"] = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return metrics
def _validate_triplet_columns(df: pd.DataFrame) -> None:
    missing = [column for column in REQUIRED_TRIPLET_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Triplet CSV is missing required columns {missing}. Found: {list(df.columns)}")
def _validate_binary_choice(series: pd.Series) -> pd.Series:
    labels = pd.to_numeric(series, errors="raise").astype(int)
    invalid = sorted(value for value in labels.unique().tolist() if value not in (0, 1))
    if invalid:
        raise ValueError(f"Choice labels must be binary 0/1. Found invalid values: {invalid}")
    return labels
def _stratified_indices(labels: np.ndarray, train_ratio: float, val_ratio: float, test_ratio: float, seed: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    train_idx: List[int] = []
    val_idx: List[int] = []
    test_idx: List[int] = []
    for label in np.unique(labels):
        cls_indices = np.where(labels == label)[0]
        rng.shuffle(cls_indices)
        n = len(cls_indices)
        n_train = int(round(n * train_ratio))
        n_val = int(round(n * val_ratio))
        n_test = n - n_train - n_val
        train_idx.extend(cls_indices[:n_train])
        val_idx.extend(cls_indices[n_train:n_train + n_val])
        test_idx.extend(cls_indices[n_train + n_val:n_train + n_val + n_test])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)
def _group_split_indices(groups: np.ndarray, train_ratio: float, val_ratio: float, seed: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    rng.shuffle(unique_groups)
    n = len(unique_groups)
    n_train = int(round(n * train_ratio))
    n_val = int(round(n * val_ratio))
    train_groups = set(unique_groups[:n_train])
    val_groups = set(unique_groups[n_train:n_train + n_val])
    test_groups = set(unique_groups[n_train + n_val:])
    all_idx = np.arange(len(groups))
    train_idx = all_idx[np.array([group in train_groups for group in groups])]
    val_idx = all_idx[np.array([group in val_groups for group in groups])]
    test_idx = all_idx[np.array([group in test_groups for group in groups])]
    return train_idx, val_idx, test_idx
def _print_split_stats(name: str, df: pd.DataFrame) -> None:
    total = len(df)
    label_counts = df["choice"].value_counts().sort_index().to_dict() if total > 0 else {}
    print(f"{name}: {total} rows | labels: {label_counts}")
def build_dataset_splits(triplets_csv: str, out_dir: str = "dataset/splits", seed: int = 42, train_ratio: float = 0.7, val_ratio: float = 0.15, test_ratio: float = 0.15, group_col: Optional[str] = None) -> Dict[str, Path]:
    ratio_sum = train_ratio + val_ratio + test_ratio
    if abs(ratio_sum - 1.0) > 1e-6:
        raise ValueError(f"Ratios must sum to 1.0, got {ratio_sum}.")
    df = pd.read_csv(triplets_csv)
    _validate_triplet_columns(df)
    clean = pd.DataFrame({
        "reference": df["reference"].astype(str),
        "candidate_a": df["candidate_a"].astype(str),
        "candidate_b": df["candidate_b"].astype(str),
        "choice": _validate_binary_choice(df["choice"]),
    })
    if group_col is not None:
        if group_col not in df.columns:
            raise ValueError(f"group_col '{group_col}' not found in CSV columns.")
        clean["group"] = df[group_col].astype(str)
        train_idx, val_idx, test_idx = _group_split_indices(groups=clean["group"].to_numpy(), train_ratio=train_ratio, val_ratio=val_ratio, seed=seed)
    else:
        train_idx, val_idx, test_idx = _stratified_indices(labels=clean["choice"].to_numpy(), train_ratio=train_ratio, val_ratio=val_ratio, test_ratio=test_ratio, seed=seed)
    train_df = clean.iloc[train_idx].reset_index(drop=True)
    val_df = clean.iloc[val_idx].reset_index(drop=True)
    test_df = clean.iloc[test_idx].reset_index(drop=True)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    train_csv = out_path / "train.csv"
    val_csv = out_path / "val.csv"
    test_csv = out_path / "test.csv"
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    _print_split_stats("train", train_df)
    _print_split_stats("val", val_df)
    _print_split_stats("test", test_df)
    return {"train": train_csv, "val": val_csv, "test": test_csv}
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def prepare_triplets_csv(images_root: str) -> pd.DataFrame:
    root = Path(images_root)
    raw_csv = root / "data.csv"
    if not raw_csv.exists():
        raise FileNotFoundError(f"Missing raw metadata: {raw_csv}")
    df = pd.read_csv(raw_csv)
    for column in ["ref_path", "left_path", "right_path"]:
        df[column] = df[column].str.replace("util/2afc_src_images/", "", regex=False)
    df = df[df["left_vote"] != df["right_vote"]].copy()
    df["choice"] = (df["right_vote"] > df["left_vote"]).astype(int)
    triplets = pd.DataFrame({
        "reference": df["ref_path"],
        "candidate_a": df["left_path"],
        "candidate_b": df["right_path"],
        "choice": df["choice"],
    })
    exists = triplets.apply(lambda row: (root / row["reference"]).exists() and (root / row["candidate_a"]).exists() and (root / row["candidate_b"]).exists(), axis=1)
    triplets = triplets[exists].reset_index(drop=True)
    out_csv = Path("dataset/processed/triplets_available.csv")
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    triplets.to_csv(out_csv, index=False)
    return triplets
def batch_ranking_loss(sim_a: torch.Tensor, sim_b: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
    sign = torch.where(label == 0, torch.ones_like(sim_a), -torch.ones_like(sim_a))
    margin = sim_a - sim_b
    return F.softplus(-sign * margin).mean()
def build_encoder_from_config(config: Dict) -> VisionTransformerEncoder:
    encoder = VisionTransformerEncoder(BackboneConfig(model_name=config["model_name"], pretrained=config["pretrained"], image_size=config["image_size"]))
    replace_layers = parse_replace_layers(config["replace_layers"])
    block_indices = resolve_block_indices(len(encoder.backbone.blocks), replace_layers)
    if config["attention_type"] != "standard":
        replace_attention_modules(encoder, make_attention_factory(config["attention_type"], top_k=config["top_k"]), block_indices=block_indices)
    return encoder
def build_optimizer_and_scheduler(encoder: nn.Module, config: Dict, steps_per_epoch: int, total_epochs: int):
    if config["attention_type"] in {"moh", "pyra", "meta"}:
        router_params = []
        base_params = []
        for name, param in encoder.named_parameters():
            if not param.requires_grad:
                continue
            if ".router." in name or ".meta_router." in name:
                router_params.append(param)
            else:
                base_params.append(param)
        param_groups = [{"params": base_params, "lr": config["lr"], "weight_decay": config["weight_decay"]}]
        if router_params:
            param_groups.append({"params": router_params, "lr": config["router_lr"], "weight_decay": config["weight_decay"]})
        optimizer = torch.optim.AdamW(param_groups)
    else:
        optimizer = torch.optim.AdamW(encoder.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    total_steps = max(1, total_epochs * max(1, steps_per_epoch))
    warmup_steps = int(total_steps * config["warmup_ratio"])
    def lr_lambda(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
        floor = config["min_lr"] / max(config["lr"], 1e-12)
        return max(floor, cosine)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler
def evaluate_loader(pipeline: DreamSimLikePipeline, loader: DataLoader, device: str, amp_enabled: bool, criterion):
    pipeline.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    with torch.no_grad():
        for batch in loader:
            ref = batch["reference"].to(device, non_blocking=True)
            a = batch["candidate_a"].to(device, non_blocking=True)
            b = batch["candidate_b"].to(device, non_blocking=True)
            label = batch["choice"].to(device, non_blocking=True)
            with autocast(enabled=amp_enabled):
                out = pipeline(ref, a, b)
                loss = criterion(out["sim_a"], out["sim_b"], label)
            total_loss += loss.item() * label.numel()
            total += label.numel()
            correct += (out["pred"] == label).sum().item()
    return total_loss / max(total, 1), correct / max(total, 1)
EXPERIMENTS: List[Dict] = [
    {"name": "baseline_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "standard", "replace_layers": None},
    {"name": "moh_all_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": "all"},
    {"name": "moh_last6_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [6, 7, 8, 9, 10, 11]},
    {"name": "moh_first6_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [0, 1, 2, 3, 4, 5]},
    {"name": "moh_every_other_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [0, 2, 4, 6, 8, 10]},
]
def format_replace_layers(value: Optional[object]) -> str:
    if value is None:
        return "none"
    if value == "all":
        return "all"
    if isinstance(value, list):
        return ",".join(str(item) for item in value)
    return str(value)
def build_encoder_for_ablation(exp: Dict) -> VisionTransformerEncoder:
    encoder = VisionTransformerEncoder(BackboneConfig(model_name=exp["model_name"], pretrained=exp["pretrained"], image_size=224))
    block_indices = resolve_block_indices(len(encoder.backbone.blocks), exp["replace_layers"])
    if exp["attention_type"] != "standard":
        replace_attention_modules(encoder, make_attention_factory(exp["attention_type"], top_k=4), block_indices=block_indices)
    return encoder
def run_ablation_experiment(exp: Dict, device: str = "cpu") -> Dict:
    encoder = build_encoder_for_ablation(exp)
    metrics = evaluate(split_csv="dataset/splits/val.csv", images_root="dataset/raw", model_name=exp["model_name"], pretrained=exp["pretrained"], image_size=224, batch_size=16, num_workers=2, device=device, max_batches=None, encoder=encoder)
    return {
        "experiment": exp["name"],
        "model_name": exp["model_name"],
        "pretrained": exp["pretrained"],
        "attention_type": exp["attention_type"],
        "replace_layers": format_replace_layers(exp["replace_layers"]),
        **metrics,
    }
def run_ablations(experiments: List[Dict], out_csv: str = "outputs/tables/ablation_results.csv", device: str = "cpu") -> pd.DataFrame:
    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    rows = [run_ablation_experiment(exp, device=device) for exp in experiments]
    if not rows:
        raise RuntimeError("No experiments configured.")
    df = pd.DataFrame(rows)
    write_header = not out_path.exists()
    mode = "a" if out_path.exists() else "w"
    with out_path.open(mode, newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        if write_header:
            writer.writeheader()
        writer.writerows(rows)
    return df

In [ ]:
class PyraAttention(nn.Module):
    def __init__(self, old_attn: nn.Module, top_k: int | None = 4):
        super().__init__()
        self.num_heads = old_attn.num_heads
        self.head_dim = old_attn.head_dim
        self.attn_dim = old_attn.attn_dim
        self.scale = old_attn.scale
        self.qkv = copy.deepcopy(old_attn.qkv)
        self.q_norm = copy.deepcopy(old_attn.q_norm)
        self.k_norm = copy.deepcopy(old_attn.k_norm)
        self.attn_drop = copy.deepcopy(old_attn.attn_drop)
        self.norm = copy.deepcopy(old_attn.norm)
        self.proj = copy.deepcopy(old_attn.proj)
        self.proj_drop = copy.deepcopy(old_attn.proj_drop)
        dim = old_attn.qkv.in_features
        self.router = nn.Linear(dim, self.num_heads)
        self.top_k = top_k

    def forward(self, x, attn_mask=None, is_causal=False):
        bsz, seq_len, _ = x.shape
        qkv = self.qkv(x).reshape(bsz, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        head_out = attn @ v
        route_logits = self.router(x.mean(dim=1))
        head_prior = torch.linspace(1.0, 0.0, self.num_heads, device=route_logits.device, dtype=route_logits.dtype)
        route_logits = route_logits + head_prior.unsqueeze(0)
        if self.top_k is not None and self.top_k < self.num_heads:
            topk_vals, topk_idx = route_logits.topk(self.top_k, dim=-1)
            masked_logits = route_logits.new_full(route_logits.shape, float("-inf"))
            masked_logits.scatter_(1, topk_idx, topk_vals)
            route_logits = masked_logits
        head_weights = route_logits.softmax(dim=-1)
        head_out = head_out * head_weights[:, :, None, None]
        out = head_out.transpose(1, 2).reshape(bsz, seq_len, self.attn_dim)
        out = self.norm(out)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out


class MetaAttention(nn.Module):
    def __init__(self, old_attn: nn.Module, top_k: int | None = 4):
        super().__init__()
        self.num_heads = old_attn.num_heads
        self.head_dim = old_attn.head_dim
        self.attn_dim = old_attn.attn_dim
        self.scale = old_attn.scale
        self.qkv = copy.deepcopy(old_attn.qkv)
        self.q_norm = copy.deepcopy(old_attn.q_norm)
        self.k_norm = copy.deepcopy(old_attn.k_norm)
        self.attn_drop = copy.deepcopy(old_attn.attn_drop)
        self.norm = copy.deepcopy(old_attn.norm)
        self.proj = copy.deepcopy(old_attn.proj)
        self.proj_drop = copy.deepcopy(old_attn.proj_drop)
        dim = old_attn.qkv.in_features
        hidden = max(32, dim // 4)
        self.meta_router = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, self.num_heads))
        self.top_k = top_k

    def forward(self, x, attn_mask=None, is_causal=False):
        bsz, seq_len, _ = x.shape
        qkv = self.qkv(x).reshape(bsz, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        head_out = attn @ v
        route_logits = self.meta_router(x.mean(dim=1))
        if self.top_k is not None and self.top_k < self.num_heads:
            topk_vals, topk_idx = route_logits.topk(self.top_k, dim=-1)
            masked_logits = route_logits.new_full(route_logits.shape, float("-inf"))
            masked_logits.scatter_(1, topk_idx, topk_vals)
            route_logits = masked_logits
        head_weights = route_logits.softmax(dim=-1)
        head_out = head_out * head_weights[:, :, None, None]
        out = head_out.transpose(1, 2).reshape(bsz, seq_len, self.attn_dim)
        out = self.norm(out)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out


def make_attention_factory(attention_type: str, top_k: int | None = 4):
    if attention_type == "standard":
        return lambda old_attn, block_idx: old_attn
    if attention_type == "moh":
        return lambda old_attn, block_idx: MoHAttention(old_attn, top_k=top_k)
    if attention_type == "pyra":
        return lambda old_attn, block_idx: PyraAttention(old_attn, top_k=top_k)
    if attention_type == "meta":
        return lambda old_attn, block_idx: MetaAttention(old_attn, top_k=top_k)
    raise ValueError(f"Unknown attention_type: {attention_type}")


def build_optimizer_and_scheduler(encoder: nn.Module, config: Dict, steps_per_epoch: int, total_epochs: int):
    if config["attention_type"] in {"moh", "pyra", "meta"}:
        router_params = []
        base_params = []
        for name, param in encoder.named_parameters():
            if not param.requires_grad:
                continue
            if ".router." in name or ".meta_router." in name:
                router_params.append(param)
            else:
                base_params.append(param)
        param_groups = [{"params": base_params, "lr": config["lr"], "weight_decay": config["weight_decay"]}]
        if router_params:
            param_groups.append({"params": router_params, "lr": config["router_lr"], "weight_decay": config["weight_decay"]})
        optimizer = torch.optim.AdamW(param_groups)
    else:
        optimizer = torch.optim.AdamW(encoder.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    total_steps = max(1, total_epochs * max(1, steps_per_epoch))
    warmup_steps = int(total_steps * config["warmup_ratio"])

    def lr_lambda(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
        floor = config["min_lr"] / max(config["lr"], 1e-12)
        return max(floor, cosine)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler


EXPERIMENTS = [
    {"name": "baseline_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "standard", "replace_layers": None},
    {"name": "moh_all_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": "all"},
    {"name": "moh_last6_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [6, 7, 8, 9, 10, 11]},
    {"name": "moh_first6_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [0, 1, 2, 3, 4, 5]},
    {"name": "moh_every_other_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [0, 2, 4, 6, 8, 10]},
    {"name": "pyra_all_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "pyra", "replace_layers": "all"},
    {"name": "meta_all_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "meta", "replace_layers": "all"},
]

print("Notebook variant override loaded: standard/moh/pyra/meta")

Notebook variant override loaded: standard/moh/pyra/meta


# Data

In [ ]:
seed_everything(CONFIG["seed"])
triplets = prepare_triplets_csv(CONFIG["images_root"])
print("usable triplets:", len(triplets))
print(triplets["choice"].value_counts(dropna=False))
splits_dir = Path("dataset/splits")
if not (splits_dir / "train.csv").exists() or not (splits_dir / "val.csv").exists() or not (splits_dir / "test.csv").exists():
    print("Building splits from dataset/processed/triplets_available.csv ...")
    build_dataset_splits(triplets_csv="dataset/processed/triplets_available.csv", out_dir=str(splits_dir), seed=CONFIG["seed"], train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)
train_ds = NightsTripletDataset(split_csv=CONFIG["train_csv"], images_root=CONFIG["images_root"], transform=build_transforms(image_size=CONFIG["image_size"], train=True))
val_ds = NightsTripletDataset(split_csv=CONFIG["val_csv"], images_root=CONFIG["images_root"], transform=build_transforms(image_size=CONFIG["image_size"], train=False))
test_ds = NightsTripletDataset(split_csv=CONFIG["test_csv"], images_root=CONFIG["images_root"], transform=build_transforms(image_size=CONFIG["image_size"], train=False))
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=CONFIG["device"].startswith("cuda"))
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=CONFIG["device"].startswith("cuda"))
print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))

# Network

In [ ]:
encoder = build_encoder_from_config(CONFIG).to(CONFIG["device"])
pipeline = DreamSimLikePipeline(encoder).to(CONFIG["device"])
optimizer, scheduler = build_optimizer_and_scheduler(encoder=encoder, config=CONFIG, steps_per_epoch=len(train_loader), total_epochs=CONFIG["epochs"])
criterion = batch_ranking_loss
amp_enabled = CONFIG["device"].startswith("cuda")
scaler = GradScaler(enabled=amp_enabled)
print("Model ready on", CONFIG["device"])
print("attention_type:", CONFIG["attention_type"], "replace_layers:", CONFIG["replace_layers"], "top_k:", CONFIG["top_k"])

# Train

In [ ]:
out_dir = Path(CONFIG["out_dir"])
(out_dir / "checkpoints").mkdir(parents=True, exist_ok=True)
(out_dir / "logs").mkdir(parents=True, exist_ok=True)
history = []
best_acc = -1.0
best_state = None
for epoch in range(1, CONFIG["epochs"] + 1):
    start = time.perf_counter()
    pipeline.train()
    train_loss_sum = 0.0
    train_count = 0
    for batch in train_loader:
        ref = batch["reference"].to(CONFIG["device"], non_blocking=True)
        a = batch["candidate_a"].to(CONFIG["device"], non_blocking=True)
        b = batch["candidate_b"].to(CONFIG["device"], non_blocking=True)
        label = batch["choice"].to(CONFIG["device"], non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=amp_enabled):
            out = pipeline(ref, a, b)
            loss = criterion(out["sim_a"], out["sim_b"], label)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss_sum += loss.item() * label.numel()
        train_count += label.numel()
    train_loss = train_loss_sum / max(train_count, 1)
    val_loss, val_acc = evaluate_loader(pipeline=pipeline, loader=val_loader, device=CONFIG["device"], amp_enabled=amp_enabled, criterion=criterion)
    epoch_seconds = time.perf_counter() - start
    row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_accuracy_2afc": val_acc, "epoch_seconds": epoch_seconds, "lr": optimizer.param_groups[0]["lr"]}
    history.append(row)
    print(f"epoch={epoch:03d} train_loss={train_loss:.5f} val_loss={val_loss:.5f} val_acc={val_acc:.4f} lr={optimizer.param_groups[0]['lr']:.2e}")
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {"epoch": epoch, "model_state_dict": pipeline.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(), "config": CONFIG, "best_val_accuracy_2afc": best_acc}
run_name = CONFIG["run_name"]
history_path = out_dir / "logs" / f"train_history_{run_name}.json"
best_path = out_dir / "checkpoints" / f"best_{run_name}.pt"
last_path = out_dir / "checkpoints" / f"last_{run_name}.pt"
torch.save({"epoch": CONFIG["epochs"], "model_state_dict": pipeline.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(), "config": CONFIG, "best_val_accuracy_2afc": best_acc}, last_path)
if best_state is not None:
    torch.save(best_state, best_path)
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
print("Saved:", history_path, best_path, last_path)

# Test

In [ ]:
val_metrics = evaluate(split_csv=CONFIG["val_csv"], images_root=CONFIG["images_root"], model_name=CONFIG["model_name"], pretrained=CONFIG["pretrained"], image_size=CONFIG["image_size"], batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"], device=CONFIG["device"], encoder=encoder)
test_metrics = evaluate(split_csv=CONFIG["test_csv"], images_root=CONFIG["images_root"], model_name=CONFIG["model_name"], pretrained=CONFIG["pretrained"], image_size=CONFIG["image_size"], batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"], device=CONFIG["device"], encoder=encoder)
print("Validation metrics:")
print(json.dumps(val_metrics, indent=2))
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    ablation_device = "cuda" if torch.cuda.is_available() else "cpu"
    ablation_df = run_ablations(EXPERIMENTS, out_csv="outputs/tables/ablation_results.csv", device=ablation_device)
    print(ablation_df)